# Orbitals, density and electrostatic potential

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mgoldey/ferric/blob/main/examples/notebooks/02_orbitals_and_esp.ipynb)

Cube files for the HOMO, LUMO and density of water, viewed with py3Dmol, and the electrostatic potential on a van der Waals surface.

Runs on a Colab CPU runtime in about a minute. Every number printed below is also asserted, so a cell that runs without an error has reproduced the reference value.

In [ ]:
%pip install -q --pre ferric py3Dmol
import os

os.environ.setdefault(
    "OPENBLAS_NUM_THREADS", "1"
)  # set before the first `import ferric`

In [ ]:
import numpy as np
import ferric

try:
    from ferric import viz
except ImportError as err:
    raise ImportError(
        "this notebook needs a ferric release that ships ferric.viz "
        "(newer than 0.1.0rc11)"
    ) from err

In [ ]:
WATER_XYZ = """3
water
O   0.000000   0.000000   0.117790
H   0.000000   0.755453  -0.471161
H   0.000000  -0.755453  -0.471161
"""
water = ferric.Molecule.from_xyz_string(WATER_XYZ, 0, 1)  # charge 0, singlet
bs = ferric.BasisSet.bundled("cc-pvdz")
rhf = ferric.run_rhf(water, bs)
assert rhf.converged
print(f"E(RHF/cc-pVDZ) = {rhf.energy:.8f} Ha")

## Orbital energies

`orbital_energies()` is in Hartree, in energy order. Water has 10 electrons, so orbital index 4 is the HOMO and 5 the LUMO.

In [ ]:
eps = np.array(rhf.orbital_energies())
homo = water.nelec() // 2 - 1
print(f"HOMO (index {homo}) = {eps[homo]:+.4f} Ha, LUMO = {eps[homo + 1]:+.4f} Ha")
assert homo == 4 and eps[homo] < 0 < eps[homo + 1]

## Cube files

`write_cube` takes **Bohr** for spacing and padding and returns the grid it used. Orbital signs are the SCF's arbitrary phase: the two colours are the two lobes, not positive and negative charge.

In [ ]:
for name, kwargs in [
    ("homo", dict(what="mo", mo_index=homo)),
    ("lumo", dict(what="mo", mo_index=homo + 1)),
    ("density", dict(what="density")),
]:
    origin, shape = ferric.write_cube(
        f"{name}.cube", water, bs, rhf, spacing=0.4, **kwargs
    )
print("grid:", shape)

# The density cube integrates to the electron count: sum(rho) * voxel volume. Water's 1s core is
# tight, so the grid sum needs a fine spacing: 0.4 Bohr gives 9.59 electrons, 0.2 Bohr gives 10.011
# (measured), so the check uses its own 0.2 Bohr grid and the picture below uses the coarse one.
ferric.write_cube("density_fine.cube", water, bs, rhf, spacing=0.2)
cube = viz.read_cube("density_fine.cube")
voxel = abs(np.linalg.det(cube["axes"]))
n_el = cube["values"].sum() * voxel
print(f"electrons on the 0.2 Bohr grid: {n_el:.3f}")
assert abs(n_el - water.nelec()) < 0.05

In [ ]:
viz.show_cube("homo.cube", mol=water, isovalue=0.05)

In [ ]:
viz.show_cube("lumo.cube", mol=water, isovalue=0.05)

In [ ]:
viz.show_cube("density.cube", mol=water, isovalue=0.05, opacity=0.6)

## Electrostatic potential on a van der Waals surface

`esp_on_surface` samples Lebedev spheres at 1.4 times the Bondi radii and returns points (Bohr) with the potential (Hartree/e). Red is negative potential (the oxygen lone-pair side), blue is positive.

In [ ]:
points, esp, n_buried = ferric.esp_on_surface(water, bs, rhf)
esp = np.array(esp)
print(f"{len(esp)} surface points, ESP from {esp.min():+.4f} to {esp.max():+.4f} Ha/e")
# The potential is most negative on the oxygen side (z > 0 is the lone-pair side in this frame)
# and most positive next to the hydrogens.
o_side = esp[points[:, 2] > 0.0].mean()
h_side = esp[points[:, 2] < -1.5].mean()
assert o_side < h_side
viz.show_surface_values(water, points, esp, vlim=0.05)

## Partial charges

The population charges and the ESP-fitted charges differ by a lot, which is why the method matters. ESP-fitted charges (CHELPG, RESP) reproduce the potential above; population charges (Mulliken, Loewdin) split the density.

In [ ]:
for name in ("mulliken_charges", "lowdin_charges", "chelpg_charges"):
    q = getattr(ferric, name)(water, bs, rhf)
    print(f"{name:16s}", np.round(q, 3), f"sum = {sum(q):+.1e}")
    assert abs(sum(q)) < 1e-6  # neutral molecule